In [1]:
using Pkg
Pkg.develop(path="D:/Library_Julia/CompEcon.jl")   
using QuantEcon,LinearAlgebra, Plots; pyplot()
using Printf,Parameters,LaTeXStrings
using CompEcon, BasisMatrices

   Resolving package versions...
     Project No packages added to or removed from `D:\notes\econ_hbc\code\quan_macro\Project.toml`
    Manifest No packages added to or removed from `D:\notes\econ_hbc\code\quan_macro\Manifest.toml`


In [ ]:
function setup(;
                β = 0.8,
                α = 2/3,
                c_e = 20.0,
                c_f = 20.0,
                ρ = 0.9,
                σ = 0.2,
                μ_z = 1.0,
                A = 0.01,
                τ = 0.1,
                N_z = 101)

    function Tauchen_Hussey(μ::Float64, σ :: Float64, ρ :: Float64, N::Int64 )

        pdf_normal(z) = exp.(-z.^2.0 ./2.0) ./ sqrt(2.0*pi)

        σ_bar = σ/sqrt(1.0 -ρ^2.0)
        θ = 1/2+ρ/4 # Floden's advice
        σ_hat = θ*σ+(1-θ)*σ_bar
        nodes, weights = QuantEcon.qnwnorm(N,μ,σ_hat^2)
        ω = pdf_normal((nodes .- μ)./σ_hat) ./ σ_hat
        p = fill(0.0, N, N)

        for i in 1:N , j in 1:N
            p[i,j] = pdf_normal((nodes[j]-(1-ρ)*μ-ρ*nodes[i])/σ)/σ
        end

        P = fill(0.0,N,N)
        for i in 1:N , j in 1:N
            P[i,j] = (p[i,j]*weights[j]/ω[j])/(sum(p[i,k]*weights[k]/ω[k] for k = 1:N))
        end

        return nodes, P
    end

    grid_Z, F = Tauchen_Hussey(μ_z, σ, ρ, N_z)
    grid_Z = exp.(grid_Z)

    # === 构造进入者的初始抽样分布
    inv_Z = F^1000
    G = inv_Z[1,:]
    
    grid_N = [collect(0:1:20); collect(22:2:100); collect(105:5:500); collect(550:50:1000); 
			 collect(1100:100:5000);  collect(5500:500:10000)]	
	N_n = length(grid_N)
	
    # === 调整成本
    adjust = fill(0.0, N_n, N_n)
    for i in 1:N_n, ip in 1:N_n 
        adjust[i,ip] = τ * max(0,grid_N[i]-grid_N[ip])
    end

    # === 生产函数
    grid_Y = fill(0.0, N_z, N_n)
    for iz in 1:N_z, iN in 1:N_n 
        grid_Y[iz,iN] = grid_Z[iz]*grid_N[iN]^α
    end

    return (β = β, α = α, grid_Z = grid_Z, grid_N = grid_N, grid_Y = grid_Y,
            adjust = adjust, F = F, G = G, c_e = c_e, c_f = c_f,
            A = A, N_z = N_z, N_n = N_n)
end

setup (generic function with 1 method)

In [4]:
params = setup()

UndefVarError: UndefVarError: `grid_z` not defined in `Main`
Suggestion: check for spelling errors or missing imports.